In [ ]:
!pip install requests beautifulsoup4 pandas lxml openpyxl


In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re


In [ ]:
def get_soup(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()
    return BeautifulSoup(r.text, "lxml")


In [ ]:
def clean_number(val):
    if not val:
        return None

    # keep digits, dots only
    val = re.findall(r"[\d,.]+", val)
    if not val:
        return None

    val = val[0].replace(",", "")
    try:
        return float(val)
    except:
        return None


In [ ]:
BASE_URL = "https://www.globalfirepower.com/countries-listing.php"
soup = get_soup(BASE_URL)

countries = []

for rec in soup.select(".recordsetContainer"):
    try:
        long_name = rec.select_one(".longFormName span").get_text(strip=True)
        iso = rec.select_one(".shortFormName span").get_text(strip=True)

        link_tag = rec.find_parent("a")
        link = "https://www.globalfirepower.com" + link_tag["href"]

        country_id = re.search(r"country_id=([^&]+)", link).group(1)

        countries.append({
            "country": long_name,
            "iso": iso,
            "country_id": country_id,
            "country_link": link
        })
    except:
        continue

base_df = pd.DataFrame(countries)
base_df.head()


,country,iso,country_id,country_link
0,United States,USA,united-states-of-america,https://www.globalfirepower.com/country-milita...
1,Russia,RUS,russia,https://www.globalfirepower.com/country-milita...
2,China,CHN,china,https://www.globalfirepower.com/country-milita...
3,India,IND,india,https://www.globalfirepower.com/country-milita...
4,South Korea,SKO,south-korea,https://www.globalfirepower.com/country-milita...


In [ ]:
other_sources = {
    'https://www.globalfirepower.com/total-population-by-country.php': 'total_population',
    'https://www.globalfirepower.com/available-military-manpower.php': 'total_military_manpower',
    'https://www.globalfirepower.com/manpower-fit-for-military-service.php': 'fit_for_service',
    'https://www.globalfirepower.com/manpower-reaching-military-age-annually.php': 'population_reaching_military_age_annually',
    'https://www.globalfirepower.com/active-military-manpower.php': 'active_personnel',
    'https://www.globalfirepower.com/active-reserve-military-manpower.php': 'reserve_personnel',
    'https://www.globalfirepower.com/manpower-paramilitary.php': 'paramilitary',
    'https://www.globalfirepower.com/aircraft-total.php': 'total_military_aircraft',
    'https://www.globalfirepower.com/aircraft-total-fighters.php': 'fighter_aircraft',
    'https://www.globalfirepower.com/aircraft-total-attack-types.php': 'attack_aircraft',
    'https://www.globalfirepower.com/aircraft-total-transports.php': 'transport_aircraft',
    'https://www.globalfirepower.com/aircraft-total-trainers.php': 'trainer_aircraft',
    'https://www.globalfirepower.com/aircraft-total-special-mission.php': 'special_mission_aircraft',
    'https://www.globalfirepower.com/aircraft-total-tanker-fleet.php': 'tanker_aircraft',
    'https://www.globalfirepower.com/aircraft-helicopters-total.php': 'total_military_helicopters',
    'https://www.globalfirepower.com/aircraft-helicopters-attack.php': 'attack_helicopters',
    'https://www.globalfirepower.com/armor-tanks-total.php': 'tanks',
    'https://www.globalfirepower.com/armor-apc-total.php': 'armored_fighting_vehicles',
    'https://www.globalfirepower.com/armor-self-propelled-guns-total.php': 'self_propelled_artillery',
    'https://www.globalfirepower.com/armor-towed-artillery-total.php': 'towed_artillery',
    'https://www.globalfirepower.com/armor-mlrs-total.php': 'rocket_projectors',
    'https://www.globalfirepower.com/navy-ships.php': 'total_naval_fleet',
    'https://www.globalfirepower.com/navy-force-by-tonnage.php': 'total_naval_fleet_tonnage_mt',
    'https://www.globalfirepower.com/navy-aircraft-carriers.php': 'aircraft_carriers',
    'https://www.globalfirepower.com/navy-helo-carriers.php': 'helicopter_carriers',
    'https://www.globalfirepower.com/navy-submarines.php': 'submarines',
    'https://www.globalfirepower.com/navy-destroyers.php': 'destroyers',
    'https://www.globalfirepower.com/navy-frigates.php': 'frigates',
    'https://www.globalfirepower.com/navy-corvettes.php': 'corvettes',
    'https://www.globalfirepower.com/navy-patrol-coastal-craft.php': 'coastal_patrol_craft',
    'https://www.globalfirepower.com/navy-mine-warfare-craft.php': 'mine_warfare_craft',
    'https://www.globalfirepower.com/defense-spending-budget.php': 'defense_budget_usd',
    'https://www.globalfirepower.com/external-debt-by-country.php': 'external_debt_usd',
    'https://www.globalfirepower.com/purchasing-power-parity.php': 'purchasing_power_parity_usd',
    'https://www.globalfirepower.com/reserves-of-foreign-exchange-and-gold.php': 'foreign_exchange_and_gold_reserves_usd',
    'https://www.globalfirepower.com/major-serviceable-airports-by-country.php': 'total_serviceable_airports',
    'https://www.globalfirepower.com/labor-force-by-country.php': 'labour_force',
    'https://www.globalfirepower.com/major-ports-and-terminals.php': 'major_ports_and_terminals',
    'https://www.globalfirepower.com/merchant-marine-strength-by-country.php': 'total_merchant_marine_fleet',
    'https://www.globalfirepower.com/railway-coverage.php': 'railway_coverage_km',
    'https://www.globalfirepower.com/roadway-coverage.php': 'roadway_coverage_km',
    'https://www.globalfirepower.com/oil-production-by-country.php': 'oil_production_bbl',
    'https://www.globalfirepower.com/oil-consumption-by-country.php': 'oil_consumption_bbl',
    'https://www.globalfirepower.com/proven-oil-reserves-by-country.php': 'proven_oil_reserves_bbl',
    'https://www.globalfirepower.com/natural-gas-production-by-country.php': 'natural_gas_production_cum',
    'https://www.globalfirepower.com/natural-gas-consumption-by-country.php': 'natural_gas_consumption_cum',
    'https://www.globalfirepower.com/proven-natural-gas-reserves-by-country.php': 'proven_natural_gas_reserves_cum',
    'https://www.globalfirepower.com/coal-production-by-country.php': 'coal_production_cum',
    'https://www.globalfirepower.com/coal-consumption-by-country.php': 'coal_consumption_mt',
    'https://www.globalfirepower.com/proven-coal-reserves-by-country.php': 'proven_coal_reserves_cum',
    'https://www.globalfirepower.com/square-land-area.php': 'total_land_area_sq_km',
    'https://www.globalfirepower.com/coastline-coverage.php': 'coastline_coverage_km',
    'https://www.globalfirepower.com/border-coverage.php': 'border_coverage_km',
    'https://www.globalfirepower.com/waterway-coverage.php': 'waterway_coverage_km'
}


In [ ]:
def scrape_metric(url, col):
    soup = get_soup(url)
    rows = []

    for rec in soup.select(".recordsetContainer"):
        try:
            country = rec.select_one(".longFormName span").get_text(strip=True)

            # extract FULL visible text
            value_block = rec.select_one(".valueContainer")
            value_text = value_block.get_text(" ", strip=True)

            rows.append({
                "country": country,
                col: clean_number(value_text)
            })
        except:
            continue

    return pd.DataFrame(rows)


In [ ]:
final_df = base_df.copy()

for url, col in other_sources.items():
    print("Scraping:", col)
    df_metric = scrape_metric(url, col)
    final_df = final_df.merge(df_metric, on="country", how="left")


Scraping: total_population
Scraping: total_military_manpower
Scraping: fit_for_service
Scraping: population_reaching_military_age_annually
Scraping: active_personnel
Scraping: reserve_personnel
Scraping: paramilitary
Scraping: total_military_aircraft
Scraping: fighter_aircraft
Scraping: attack_aircraft
Scraping: transport_aircraft
Scraping: trainer_aircraft
Scraping: special_mission_aircraft
Scraping: tanker_aircraft
Scraping: total_military_helicopters
Scraping: attack_helicopters
Scraping: tanks
Scraping: armored_fighting_vehicles
Scraping: self_propelled_artillery
Scraping: towed_artillery
Scraping: rocket_projectors
Scraping: total_naval_fleet
Scraping: total_naval_fleet_tonnage_mt
Scraping: aircraft_carriers
Scraping: helicopter_carriers
Scraping: submarines
Scraping: destroyers
Scraping: frigates
Scraping: corvettes
Scraping: coastal_patrol_craft
Scraping: mine_warfare_craft
Scraping: defense_budget_usd
Scraping: external_debt_usd
Scraping: purchasing_power_parity_usd
Scraping: f

In [ ]:
# keep NaN while debugging
final_df.isna().sum().sort_values(ascending=False)

final_df.sort_values("country", inplace=True)

final_df.to_excel("global_firepower_complete_dataset.xlsx", index=False)
final_df.head()


,country,iso,country_id,country_link,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,...,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km
120,Afghanistan,AFG,afghanistan,https://www.globalfirepower.com/country-milita...,40121552.0,15647405.0,8826741.0,842553.0,75000.0,0.0,...,8.020000e+07,8.020000e+07,4.955400e+10,767000.0,503000.0,66000000.0,652230.0,0.0,5987.0,1200.0
76,Albania,ALB,albania,https://www.globalfirepower.com/country-milita...,3107100.0,1522479.0,1292554.0,62142.0,7500.0,0.0,...,5.062300e+07,5.062300e+07,5.692000e+09,473000.0,255000.0,522000000.0,28748.0,362.0,691.0,41.0
26,Algeria,ALG,algeria,https://www.globalfirepower.com/country-milita...,47022473.0,22570787.0,19185169.0,752360.0,130000.0,150000.0,...,1.007260e+11,4.796300e+10,4.504000e+12,0.0,3000.0,233000000.0,2381740.0,998.0,6734.0,0.0
58,Angola,ANG,angola,https://www.globalfirepower.com/country-milita...,37202061.0,7440412.0,3720206.0,372021.0,107000.0,0.0,...,5.514000e+09,1.397000e+09,3.430020e+11,0.0,0.0,0.0,1246700.0,1600.0,5369.0,1300.0
31,Argentina,ARG,argentina,https://www.globalfirepower.com/country-milita...,46994384.0,20677529.0,17575900.0,704916.0,108000.0,12370.0,...,4.328000e+10,4.622800e+10,3.964640e+11,869000.0,2534000.0,799999000.0,2780400.0,4989.0,11968.0,11000.0


In [ ]:
numeric_cols = final_df.select_dtypes(include="number").columns
final_df[numeric_cols] = final_df[numeric_cols].fillna(0)
